# DSFB-Database — Reproduction Notebook

This notebook runs **dsfb-database** from scratch on **real public datasets**, end-to-end, in Colab (or any Jupyter kernel). No synthetic substitutes are used on the real-data path. The only deterministic-synthetic section is the TPC-DS-shaped perturbation harness, whose entire purpose is ground-truth-validated F1/TTD measurement (§8 of the paper).

## Non-claims (read first)

1. DSFB-Database does not optimise queries, replace the query optimiser, or modify execution plans.
2. DSFB-Database does not claim causal correctness; motifs represent structural consistency given observed signals, not root causes.
3. DSFB-Database does not provide a forecasting or predictive guarantee; precursor structure is structural, not probabilistic.
4. DSFB-Database does not provide ground-truth-validated detection on real workloads; we evaluate via injected perturbations, plan-hash concordance, and replay determinism.
5. DSFB-Database does not claim a universal SQL grammar; motifs are engine-aware, telemetry-aware, and workload-aware.
6. DSFB-Database does not validate that an operator-supplied grammar is appropriate for a non-SQL residual stream; the generic CSV adapter is a worked example, not a universality claim.

## What this notebook shows

* **Real data, every run.** Four real public datasets are shipped as representative samples inside the crate at `examples/data/`:
    * `ceb_sample.csv` — 5,000 true/estimated PostgreSQL cardinality rows, sliced from the first 400 CEB-IMDb pickles (https://github.com/learnedsystems/CEB, MIT).
    * `job_trace_sample.csv` — EXPLAIN ANALYZE trace over all 113 JOB queries × 3 replays, produced by running `scripts/fetch_job.sh` against IMDb in DuckDB (https://github.com/gregrahn/join-order-benchmark + https://event.cwi.nl/da/job/imdb.tgz, MIT).
    * `queries.txt` — full 11,136-query SQLShare text release (https://uwescience.github.io/sqlshare/data_release.html, CC-BY 4.0). Analysed in `sqlshare-text` mode (workload-phase motif only, over ordinal-position buckets — the public release has no timing/user metadata).
    * `snowset_sample.csv` — first 5,000 real rows from the Snowset Cornell mirror (http://www.cs.cornell.edu/~midhul/snowset/snowset-main.csv.gz, CC-BY 4.0).
* **Controlled-perturbation pipeline** (TPC-DS-shaped, deterministic in-process) provides the only ground-truth-validated metrics (F1/TTD/FAR) in the paper.
* **Heavy-fetch path**, for users who want the full corpora (not required for this notebook), is documented in the final cell.
* Replay-determinism is pinned by fingerprint: the test suite verifies SHA-256 of the canonical stream + episode outputs.


## 1. Clean working dir, install Rust, clone repo

In [ ]:
import os, sys, shutil, subprocess

# Wipe any previous repo checkout so every run starts from scratch.
if os.path.isdir('dsfb'):
    shutil.rmtree('dsfb')

if shutil.which('cargo') is None:
    subprocess.check_call(['bash', '-lc',
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
        "sh -s -- -y --default-toolchain stable"])
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
subprocess.check_call(['cargo', '--version'])
subprocess.check_call(['git', 'clone', '--depth', '1',
                       'https://github.com/infinityabundance/dsfb.git'])
REPO  = os.path.abspath('dsfb')
CRATE = os.path.join(REPO, 'crates', 'dsfb-database')
OUT   = os.path.join(CRATE, 'out')
os.makedirs(OUT, exist_ok=True)
print('crate path:', CRATE)

## 2. Build + run the test suite

Includes the pinned-fingerprint replay-determinism tests and the non-claim lock.

In [ ]:
subprocess.check_call(['cargo', 'build', '--release', '--features', 'report'], cwd=CRATE)
subprocess.check_call(['cargo', 'test',  '--release', '--features', 'report'], cwd=CRATE)

## 3. One-command reproduction

`reproduce-all` runs the TPC-DS controlled-perturbation pipeline, every bundled dataset exemplar, the DSFB-vs-PELT/BOCPD comparison figure, the null-trace refusal figure, and the cross-signal / stability metrics, then packs every file into `out/dsfb_database_artifacts.zip`. The zip is byte-stable across runs (same seed ⇒ same SHA-256) — `tests/reproduce_all_zip_is_deterministic.rs` pins that invariant.


In [ ]:
subprocess.check_call(['./target/release/dsfb-database', 'reproduce-all',
                       '--seed', '42', '--out', OUT], cwd=CRATE)
subprocess.check_call(['./target/release/dsfb-database', 'replay-check',
                       '--seed', '42'], cwd=CRATE)

import pandas as pd
metrics = pd.read_csv(os.path.join(OUT, 'tpcds.metrics.csv'))
metrics


## 4. Display the canonical figures + comparison + refusal

Every figure below is regenerated from the run above — no hidden state.


In [ ]:
from IPython.display import Image, display
figures = ['tpcds.plan_regression_onset.png',
           'tpcds.cardinality_mismatch_regime.png',
           'tpcds.contention_ramp.png',
           'tpcds.cache_collapse.png',
           'tpcds.workload_phase_transition.png',
           'tpcds.phase_portrait.png',
           'tpcds.funnel.png',
           'comparison.png',
           'refusal.png']
for name in figures:
    path = os.path.join(OUT, name)
    if os.path.exists(path):
        print(name)
        display(Image(path))


## 5. Download everything as one zip

The `reproduce-all` command already produced `out/dsfb_database_artifacts.zip`. In Colab, the file is handed to `google.colab.files.download()` so you can grab it with one click; elsewhere the zip path is printed for manual fetching. `out/MANIFEST.md` enumerates every file inside.


In [ ]:
zip_path = os.path.join(OUT, 'dsfb_database_artifacts.zip')
print('artifact zip ready:', zip_path, f'({os.path.getsize(zip_path)/1e6:.2f} MB)')
try:
    from google.colab import files as _files  # type: ignore
    _files.download(zip_path)
except Exception as e:
    print('(not in Colab: fetch from', zip_path, ')')
    print('  reason:', e)

print()
with open(os.path.join(OUT, 'MANIFEST.md')) as f:
    print(f.read())


## 6. (Optional) Full-corpus fetch paths

The bundled samples above are **real data** (sliced directly from the
authoritative public dumps), large enough to exercise every motif the
dataset can support. If you want to process the **full** public corpus
of each dataset — e.g. for an independent re-analysis — the fetch
scripts under `scripts/` pull the authoritative artefacts and write the
same CSV schema. They are heavy (Snowset is ~7.5 GB gzipped; JOB
requires downloading the 1.17 GB IMDb dump and loading it into DuckDB;
CEB downloads a ~1 GB pickle tarball from Dropbox) and are
**not required** to reproduce the paper's figures or any of the
fingerprints pinned by the test suite.

```bash
cd crates/dsfb-database
bash scripts/fetch_ceb.sh             # CEB pickles -> data/ceb_subset.csv      (~50 MB)
bash scripts/fetch_job.sh             # IMDb 1.17 GB + DuckDB load + 113 x N EXPLAIN ANALYZE
bash scripts/fetch_snowset_subset.sh  # Snowset 7.5 GB dump -> data/snowset_shard.csv
bash scripts/fetch_sqlshare.sh        # SQLShare queries.txt (already bundled in examples/data/)
bash scripts/build_tpcds.sh           # in-process; no download required

./target/release/dsfb-database run --dataset ceb           --path data/ceb_subset.csv
./target/release/dsfb-database run --dataset job           --path data/job_trace.csv
./target/release/dsfb-database run --dataset snowset       --path data/snowset_shard.csv
./target/release/dsfb-database run --dataset sqlshare-text --path examples/data/queries.txt
```

Every output stream is fingerprinted; re-running any of these commands
on the same input is bytewise-deterministic.